In [ ]:
import spacy
import os

In [ ]:
# CAN YOU SKIP THIS???
# After you download a model into your virtual environment for the first time, you can comment out the download line.
# spaCy's medium and large models will give us the best results for NLP tagging.
# nlp = spacy.cli.download("en_core_web_sm")
nlp = spacy.cli.download("en_core_web_md")
# nlp = spacy.cli.download("en_core_web_lg")


File Collection

In [ ]:
collPath = 'text-files'
for file in os.listdir(collPath):
    if file.endswith(".txt"):
        filepath = f"{collPath}/{file}"
        name, extension = os.path.splitext(file)
        print(name)
        with open(filepath, 'r', encoding='utf8') as f:
            readFile = f.read()
            lengthFile = len(readFile)
            print(lengthFile)

Collection of Words

In [ ]:
nlp = spacy.load("en_core_web_md")
# ebb: You have to redefine nlp to **load** the spacy model!

for file in os.listdir(collPath):
    if file.endswith(".txt"):
        filepath = f"{collPath}/{file}"
        name, extension = os.path.splitext(file)
        print(name)
        with open(filepath, 'r', encoding='utf8') as f:
            readFile = f.read()
            spacyRead = nlp(readFile)
            for token in spacyRead:
                print(token.text, "---->", token.pos_, ":::::", token.lemma_)                                

Selecting some data

In [ ]:
def wordCollector(words, unit):
    wordList = []
    for token in words:
        if token.pos_ == "ADJ":
            wordList.append((token.lemma_, unit))
    uniqueLems = set(wordList)
    return uniqueLems

for file in os.listdir(collPath):
    if file.endswith(".txt"):
        filepath = f"{collPath}/{file}"
        name, extension = os.path.splitext(file)
        with open(filepath, 'r', encoding='utf8') as f:
            readFile = f.read()
            spacyRead = nlp(readFile)
            myWords = wordCollector(spacyRead, name)
            print(myWords)


TSV file

In [ ]:
import pandas as pd
import nltk
from nltk.corpus import wordnet as wn

def wordCollector(words, unit):
    wordList = []
    nodeAtts = []
    synsetCounts = []
    unitList = []
    for token in words:
        if token.pos_ == "ADJ":
            synsets = len(wn.synsets(token.lemma_))
            wordList.append(token.lemma_)
            nodeAtts.append(token.pos_)
            synsetCounts.append(synsets)
            unitList.append(unit)

    data = {
        'word': wordList,
        'nodeType': nodeAtts,
        'synsetCount': synsetCounts,
        'unit': unitList
    }
    df = pd.DataFrame(data)

    # ebb: Trying to simplify your TSV data. We want to COUNT the word occurrences, keep the other columns (they're uniform per lemma).
    df = (df.groupby(['word', 'nodeType', 'synsetCount', 'unit'], as_index=False)
            .size()   # ebb: .size() will deliver the number of times a word is used in the file!
            .rename(columns={'size': 'count'})) # ebb: This gives you a new column for the count of the words in the file.
        # ebb: We can use that for the edge thickness: count of the times a word is used. Should simplify your network visualization!
    return df
   
allDataFrames = []

for file in os.listdir(collPath):
    if file.endswith(".txt"):
        filepath = f"{collPath}/{file}"
        name, extension = os.path.splitext(file)
        with open(filepath, 'r', encoding='utf8') as f:
            readFile = f.read()
            spacyRead = nlp(readFile)
            myDataFrame = wordCollector(spacyRead, name)
           
            allDataFrames.append(myDataFrame)

outputFilePath = 'networkData-Simpler.tsv'

fullDataFrame = pd.concat(allDataFrames, ignore_index=True)


fullDataFrame.to_csv(outputFilePath, sep='\t', index=False)
print('I just saved a NEW dataframe as a NEW TSV file.')
